# Deploying Machine Learning Models on Hugging Face Hub with Gradio Integration

Run in order with Python 3.11 and a real CPU. Downloads the full hash-verified official BLIP checkpoint and the public model-card demo image, then tests an actual loopback Gradio server and authenticated local endpoint. No public share tunnel, Space creation, HF credential or paid API is needed. The private-Space helper is defined and its missing-token path is checked, but hosted private-Hub permissions are not tested here. Use a read-scoped HF_TOKEN in your own environment for that optional deployment.

```bash
python -m pip install torch==2.10.0 transformers==5.22.0 gradio==6.31.0 gradio_client==2.7.2 huggingface-hub==2.4.0 pillow==10.4.0 requests==2.34.2
```

In [ ]:
import os, tempfile, hashlib, io, secrets, socket, importlib.util, sys
from pathlib import Path
from importlib.metadata import version
from unittest.mock import patch
import requests
import torch
from PIL import Image
import gradio as gr
from gradio_client import Client, handle_file
torch.set_num_threads(2)
work = tempfile.TemporaryDirectory(prefix='tdb-blip-deployment-')
work_path = Path(work.name)
print({p: version(p) for p in ['torch', 'transformers', 'gradio', 'gradio_client']})
print('Local-only demonstration: no public tunnel, account upload or hosted Space.')

In [ ]:
APP_SOURCE = 'import os\nimport tempfile\nimport hashlib\nfrom pathlib import Path\nfrom functools import lru_cache\nimport torch\nfrom PIL import Image\nimport gradio as gr\nfrom transformers import BlipProcessor, BlipForConditionalGeneration\nfrom huggingface_hub import snapshot_download\n\nMODEL_ID = "Salesforce/blip-image-captioning-base"\nREVISION = "82a37760796d32b1411fe092ab5d4e227313294b"\nWEIGHT_SHA256 = "d6638651a5526cc2ede56f2b5104d6851b0755816d220e5e046870430180c767"\n\n@lru_cache(maxsize=1)\ndef load_captioner():\n    checkpoint = Path(snapshot_download(MODEL_ID, revision=REVISION, token=False,\n        cache_dir=Path(tempfile.gettempdir()) / "tdb-archive27-model-cache",\n        allow_patterns=["*.json", "*.txt", "pytorch_model.bin"]))\n    with (checkpoint / "pytorch_model.bin").open("rb") as handle:\n        if hashlib.file_digest(handle, "sha256").hexdigest() != WEIGHT_SHA256:\n            raise RuntimeError("Unexpected official checkpoint bytes")\n    processor = BlipProcessor.from_pretrained(checkpoint, local_files_only=True)\n    model = BlipForConditionalGeneration.from_pretrained(checkpoint,\n        local_files_only=True, dtype=torch.float32).to("cpu").eval()\n    return processor, model\n\ndef caption_image(image):\n    if not isinstance(image, Image.Image):\n        raise ValueError("Upload an image first")\n    if image.width < 1 or image.height < 1 or image.width * image.height > 4_000_000:\n        raise ValueError("Use an image with at most four million pixels")\n    image = image.convert("RGB")\n    processor, model = load_captioner()\n    inputs = processor(images=image, return_tensors="pt")\n    with torch.inference_mode():\n        tokens = model.generate(**inputs, max_new_tokens=40, do_sample=False, num_beams=1)\n    result = processor.decode(tokens[0], skip_special_tokens=True).strip()\n    if not result:\n        raise RuntimeError("The model returned an empty caption")\n    return result\n\ndef build_demo():\n    return gr.Interface(fn=caption_image,\n        inputs=gr.Image(type="pil", image_mode="RGB", sources=["upload"], label="Upload an image"),\n        outputs=gr.Textbox(label="Caption"), api_name="caption",\n        title="BLIP Image Captioning", flagging_mode="never").queue(default_concurrency_limit=1, max_size=8)\n\nif __name__ == "__main__":\n    host = "0.0.0.0" if os.getenv("SPACE_ID") else "127.0.0.1"\n    build_demo().launch(server_name=host, server_port=int(os.getenv("PORT", "7860")), share=False)\n'
REQUIREMENTS = 'torch==2.10.0\ntransformers==5.22.0\ngradio==6.31.0\ngradio_client==2.7.2\nhuggingface-hub==2.4.0\npillow==10.4.0\n'
app_path = work_path / 'app.py'
app_path.write_text(APP_SOURCE)
(work_path / 'requirements.txt').write_text(REQUIREMENTS)
spec = importlib.util.spec_from_file_location('tdb_blip_demo', app_path)
app_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_module)
caption_image = app_module.caption_image
build_demo = app_module.build_demo
print(APP_SOURCE)
print('requirements.txt:\n' + REQUIREMENTS)

In [ ]:
DEMO_IMAGE_URL = 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg'
DEMO_IMAGE_SHA256 = '9eeaa87013b4e800930e8a411b58ff9e2fd5383906b1a022f4a712720af34cc2'
response = requests.get(DEMO_IMAGE_URL, timeout=30)
response.raise_for_status()
assert len(response.content) <= 4_000_000
assert hashlib.sha256(response.content).hexdigest() == DEMO_IMAGE_SHA256
image_path = work_path / 'demo.jpg'
image_path.write_bytes(response.content)
with Image.open(io.BytesIO(response.content)) as opened:
    image = opened.convert('RGB')
print('Official BLIP demo image:', image.size, image.mode)
print('Input SHA-256:', DEMO_IMAGE_SHA256)

In [ ]:
caption = caption_image(image)
processor, model = app_module.load_captioner()
with torch.inference_mode():
    independent_tokens = model.generate(**processor(images=image, return_tensors='pt'),
        max_new_tokens=40, do_sample=False, num_beams=1)
independent_caption = processor.decode(independent_tokens[0], skip_special_tokens=True).strip()
assert caption == independent_caption and caption
print('Actual CPU caption:', caption)
print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
def unused_local_port():
    with socket.socket() as sock:
        sock.bind(('127.0.0.1', 0))
        return sock.getsockname()[1]
demo = build_demo()
port = unused_local_port()
try:
    demo.launch(server_name='127.0.0.1', server_port=port, share=False, prevent_thread_lock=True)
    url = f'http://127.0.0.1:{port}'
    local_client = Client(url, verbose=False)
    api_caption = local_client.predict(handle_file(str(image_path)), api_name='/caption')
    assert api_caption == caption
    print('Actual Gradio HTTP caption:', api_caption)
    assert requests.get(url + '/config', timeout=15).status_code == 200
finally:
    demo.close()
print('Local Gradio app closed.')

In [ ]:
# Private Hugging Face Spaces use a read token resolved locally, not a public share URL.
def private_space_client(space_id):
    import re
    if not re.fullmatch(r'[A-Za-z0-9_-]+/[A-Za-z0-9_-]+', space_id):
        raise ValueError('Use your own owner/space-name')
    token = os.getenv('HF_TOKEN')
    if not token:
        raise RuntimeError('Set a read-scoped HF_TOKEN locally')
    return Client(space_id, token=token)

with patch.dict(os.environ):
    os.environ.pop('HF_TOKEN', None)
    try:
        private_space_client('your-account/your-private-space')
    except RuntimeError:
        print('Missing private-Hub token rejected before any hosted request.')
    else:
        raise AssertionError('Missing token accepted')

# Separately test actual local Gradio authentication, not hosted Hub permissions.
test_user, test_password = 'tutorial-test', secrets.token_urlsafe(24)
secured_demo = build_demo()
secure_port = unused_local_port()
try:
    secured_demo.launch(server_name='127.0.0.1', server_port=secure_port, share=False,
        prevent_thread_lock=True, auth=(test_user, test_password))
    secure_url = f'http://127.0.0.1:{secure_port}'
    anonymous = requests.get(secure_url + '/config', timeout=15)
    assert anonymous.status_code in (401, 403)
    authorized = Client(secure_url, auth=(test_user, test_password), verbose=False)
    private_caption = authorized.predict(handle_file(str(image_path)), api_name='/caption')
    assert private_caption == caption
    print('Anonymous local config rejected:', anonymous.status_code)
    print('Authenticated local Gradio caption:', private_caption)
finally:
    secured_demo.close()
print('Local authentication fixture closed; no Hugging Face Space was created.')

In [ ]:
for invalid in [None, 'not an image', Image.new('RGB', (2100, 2100))]:
    try:
        caption_image(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid image accepted')
assert not (work_path / '.env').exists()
assert 'share=False' in APP_SOURCE and 'token=False' in APP_SOURCE
assert api_caption == private_caption == independent_caption
print('PASS: full official BLIP, independent token decode, real Gradio HTTP upload/prediction, local auth boundary and invalid-image checks.')
del model, processor
app_module.load_captioner.cache_clear()
work.cleanup()
print('Owned app/input directory removed; reusable model cache retained.')

## Original source-code and transcript appendix

Nonexecuting original archive material; use the maintained cells above.

```text
!pip install transformers
!pip install gradio
!pip install gradio_client
```

```text
import os
import gradio as gr
from transformers import pipeline
```

```text
pipe = pipeline("image-to-text",
                model="./models/Salesforce/blip-image-captioning-base")
```

```text
def launch(input):
    out = pipe(input)
    return out[0]['generated_text']
```

```text
iface = gr.Interface(launch,
                     inputs=gr.Image(type='pil'),
                     outputs="text")
```

```text
iface.launch(share=True, 
             server_port=int(os.environ['PORT1']))
```

```text
iface.close()
```

```text
import gradio as gr
from transformers import pipeline

pipe = pipeline("image-to-text",
                model="Salesforce/blip-image-captioning-base")

def launch(input):
    out = pipe(input)
    return out[0]['generated_text']
iface = gr.Interface(launch,
                     inputs=gr.Image(type='pil'),
                     outputs="text")
iface.launch()
```

```text
from gradio_client import Client

client = Client("eddyS/blip-image-captioning-2")
result = client.predict(
        "image.jpg",    # filepath  in 'input' Image component
        api_name="/predict"
)
print(result)
```

```text
from gradio_client import Client

client = Client("eddyS/blip-image-captioning-2",
                hf_token=hf_access_token
               )
result = client.predict(
  "kittens.jpg",
  api_name="/predict"
)
print(result)
# client = Client("abidlabs/whisper-large-v2", 
)
```

```text
HF_TOKEN="abc1234" # not recommended
```

```text
HF_ACCESS_TOKEN="abc123"
```

```text
# !pip install python-dotenv # install library
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())
hf_access_token = os.getenv("HF_ACCESS_TOKEN")
```